# Sales Forecasting on Walmart Store
#### **Time Series Analysis**

**Dataset:** Walmart Store Sales Forecasting (Kaggle)  

**Problem Statement:** Since Walmart's weekly sales are heavily influenced by seasonality — holiday weeks in particular — stores risk poor inventory and staffing decisions without an accurate forecasting system. This project aims to predict weekly sales from historical patterns, evaluated using WMAE.

**Tools:** Python -- Pandas -- Statsmodels -- openpyxl (Excel)  

*WMAE = Walmart's official metric that weights holiday-week accuracy 5x higher*
*Train/test split is Chronological*

# 1. Setup + Data Loading
Import libraries and load the three raw files: `train.csv` (historical weekly sales), `stores.csv` (store metadata), and `features.csv` (external factors like fuel price, CPI, unemployment, markdowns).

In [1]:
import sys
!{sys.executable} -m pip install pandas numpy scikit-learn statsmodels openpyxl


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load datasets
train = pd.read_csv('data/train.csv')
stores = pd.read_csv('data/stores.csv')
features = pd.read_csv('data/features.csv')

print(f'train.csv    : {train.shape[0]:,} rows x {train.shape[1]} columns')
print(f'stores.csv   : {stores.shape[0]:,} rows x {stores.shape[1]} columns')
print(f'features.csv : {features.shape[0]:,} rows x {features.shape[1]} columns')

train.csv    : 421,570 rows x 5 columns
stores.csv   : 45 rows x 3 columns
features.csv : 8,190 rows x 12 columns


In [3]:
from IPython.display import display

print('train.csv:')
display(train.head())

print('stores.csv:')
display(stores.head())

print('features.csv:')
display(features.head())

train.csv:


,Store,Dept,Date,Weekly_Sales,IsHoliday
0,1,1,2010-02-05,24924.50,False
1,1,1,2010-02-12,46039.49,True
2,1,1,2010-02-19,41595.55,False
3,1,1,2010-02-26,19403.54,False
4,1,1,2010-03-05,21827.90,False


stores.csv:


,Store,Type,Size
0,1,A,151315
1,2,A,202307
2,3,B,37392
3,4,A,205863
4,5,B,34875


features.csv:


,Store,Date,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday
0,1,2010-02-05,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,False
1,1,2010-02-12,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,True
2,1,2010-02-19,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,False
3,1,2010-02-26,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,False
4,1,2010-03-05,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,False


In [4]:
for name, data in [('train', train), ('stores', stores), ('features', features)]:
    print(f'--- {name}.csv ---')
    print(f'Nulls:\n{data.isnull().sum()[data.isnull().sum() > 0]}')
    print(f'Duplicate rows: {data.duplicated().sum()}')
    print()

--- train.csv ---
Nulls:
Series([], dtype: int64)
Duplicate rows: 0

--- stores.csv ---
Nulls:
Series([], dtype: int64)
Duplicate rows: 0

--- features.csv ---
Nulls:
MarkDown1       4158
MarkDown2       5269
MarkDown3       4577
MarkDown4       4726
MarkDown5       4140
CPI              585
Unemployment     585
dtype: int64
Duplicate rows: 0



### Observation result

- `train.csv` has weekly sales at the **Store–Dept–Date** level 
- `features.csv` has around 5k missing values in `MarkDown1`–`MarkDown5` [they only started partway during data collection period]
- `Date` columns are stored as text/object -> convert to `datetime`

*`stores.csv` is joined onto `train.csv` using Store as the key, so `Type` and `Size` become available on every transaction row.*

# 2. Merge + Clean Datasets

Process:
1. Merge *train.csv* with *store.csv* (`Type`, `Size`) and *features.csv* (`Temperature`, `Fuel_Price`, `CPI`, `Unemployment`, `MarkDown1`–`5`)
2. Convert `Date` to datetime and sort chronologically **[ordering matters for everything downstream]**

In [5]:
# Merge train with store
df = train.merge(stores, on='Store', how='left')

# Merge with features (drop duplicate 'IsHoliday' column from features before merging)
df = df.merge(
    features.drop(columns=['IsHoliday']),
    on=['Store', 'Date'],
    how='left'
)

# Convert Date to datetime and sort by Store, Dept, Date
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)

# Fill missing markdowns with 0 (no promotion running that week)
markdown_cols = [c for c in df.columns if 'MarkDown' in c]
df[markdown_cols] = df[markdown_cols].fillna(0)

print(f'Merged dataset: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Date range: {df["Date"].min().date()} -> {df["Date"].max().date()}')
df.head()

Merged dataset: 421,570 rows x 16 columns
Date range: 2010-02-05 -> 2012-10-26


,Store,Dept,Date,Weekly_Sales,IsHoliday,Type,Size,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment
0,1,1,2010-02-05,24924.50,False,A,151315,42.31,2.572,0.0,0.0,0.0,0.0,0.0,211.096358,8.106
1,1,1,2010-02-12,46039.49,True,A,151315,38.51,2.548,0.0,0.0,0.0,0.0,0.0,211.242170,8.106
2,1,1,2010-02-19,41595.55,False,A,151315,39.93,2.514,0.0,0.0,0.0,0.0,0.0,211.289143,8.106
3,1,1,2010-02-26,19403.54,False,A,151315,46.63,2.561,0.0,0.0,0.0,0.0,0.0,211.319643,8.106
4,1,1,2010-03-05,21827.90,False,A,151315,46.50,2.625,0.0,0.0,0.0,0.0,0.0,211.350143,8.106


In [6]:
# Make time based column (year, month, week)
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Week'] = df['Date'].dt.isocalendar().week

df[['Date', 'Year', 'Month', 'Week']].head()

,Date,Year,Month,Week
0,2010-02-05,2010,2,5
1,2010-02-12,2010,2,6
2,2010-02-19,2010,2,7
3,2010-02-26,2010,2,8
4,2010-03-05,2010,3,9


### Observation result

- Dataset spans **Feb 2010 – Oct 2012** [weekly frequency (Fridays)]

# 3. Exploratory Data Analysis (EDA)

*Explore overall sales trend, holiday effect, and differences across store types -- data visualization is done in excel*

In [7]:
# Aggregate total weekly sales across all stores/departments
weekly_sales = df.groupby('Date')['Weekly_Sales'].sum().reset_index()
weekly_sales.columns = ['Date', 'Total_Weekly_Sales']

print(weekly_sales['Total_Weekly_Sales'].describe().round(0))
weekly_sales.head()

count         143.0
mean     47113419.0
std       5444206.0
min      39599853.0
25%      44880588.0
50%      46243900.0
75%      47792025.0
max      80931416.0
Name: Total_Weekly_Sales, dtype: float64


,Date,Total_Weekly_Sales
0,2010-02-05,49750740.50
1,2010-02-12,48336677.63
2,2010-02-19,48276993.78
3,2010-02-26,43968571.13
4,2010-03-05,46871470.30


In [8]:
# Holiday vs non-holiday effect
holiday_avg = df.groupby('IsHoliday')['Weekly_Sales'].mean().round(0)
print('Average weekly sales (Holiday vs Non-Holiday):')
lift = (holiday_avg[True] / holiday_avg[False] - 1) * 100
holiday_avg = df.groupby('IsHoliday')['Weekly_Sales'].mean().round(0).reset_index()
holiday_avg.columns = ['IsHoliday', 'Avg_Weekly_Sales']
print(holiday_avg)
print(f'\nHoliday weeks see {lift:.1f}% higher average sales per Store-Dept')


Average weekly sales (Holiday vs Non-Holiday):
   IsHoliday  Avg_Weekly_Sales
0      False           15901.0
1       True           17036.0

Holiday weeks see 7.1% higher average sales per Store-Dept


In [9]:
# Looking at those weeks with top weekly_sales
df['Week'] = df['Date'].dt.isocalendar().week

weekly_avg = df.groupby('Week')['Weekly_Sales'].mean().round(0).reset_index()
weekly_avg.columns = ['Week', 'Avg_Weekly_Sales']
weekly_avg.sort_values('Avg_Weekly_Sales', ascending=False).head(10)

,Week,Avg_Weekly_Sales
50,51,26396.0
46,47,22221.0
49,50,20413.0
48,49,18669.0
21,22,16780.0
26,27,16715.0
47,48,16709.0
13,14,16547.0
22,23,16507.0
6,7,16485.0


In [10]:
#Looking at month with top weekly_sales
df['Month'] = df['Date'].dt.month

monthly_avg = df.groupby('Month')['Weekly_Sales'].mean().round(0).reset_index()
monthly_avg.columns = ['Month', 'Avg_Weekly_Sales']
monthly_avg.sort_values('Avg_Weekly_Sales', ascending=False).head(10)

,Month,Avg_Weekly_Sales
11,12,19356.0
10,11,17491.0
5,6,16326.0
7,8,16063.0
1,2,16009.0
6,7,15861.0
4,5,15776.0
3,4,15650.0
2,3,15417.0
9,10,15244.0


In [11]:
# Summary by store type
type_summary = df.groupby('Type').agg(
    Avg_Weekly_Sales=('Weekly_Sales', 'mean'),
    Total_Sales=('Weekly_Sales', 'sum'),
    Avg_Store_Size=('Size', 'mean'),
    Num_Stores=('Store', lambda x: x.nunique())
).round(0)

type_summary

,Avg_Weekly_Sales,Total_Sales,Avg_Store_Size,Num_Stores
Type,,,,
A,20100.0,4.331015e+09,182231.0,22
B,12237.0,2.000701e+09,101819.0,17
C,9520.0,4.055035e+08,40536.0,6


### 📝 Observations — Step 3

- Clear seasonal spikes are expected around **pre-Christmas** [Week 51 (~18-24 Dec), avg ~26,396] and **Thanksgiving** [Week 47 (~20-26 Nov), avg ~22,221]
- **December (month 12)** has the highest average weekly sales, followed by **November**
- Store **Type A** tends to be the largest and the highest revenue format; **Type C** is the smallest

# 4. Chronological Train/Test Split

*pick a cutoff **date**: everything before it is `train`, everything after is `test`.*

In [12]:
# Use total weekly sales across all stores as the target series
ts = weekly_sales.set_index('Date')['Total_Weekly_Sales'].asfreq('W-FRI')

# Chronological split (last ~20% of weeks as test)
split_idx = int(len(ts) * 0.8)
train_ts = ts.iloc[:split_idx]
test_ts = ts.iloc[split_idx:]

print(f'Train: {train_ts.index.min().date()} -> {train_ts.index.max().date()}  ({len(train_ts)} weeks)')
print(f'Test : {test_ts.index.min().date()} -> {test_ts.index.max().date()}  ({len(test_ts)} weeks)')

Train: 2010-02-05 -> 2012-04-06  (114 weeks)
Test : 2012-04-13 -> 2012-10-26  (29 weeks)


#### **Test Set Limitation**: Does Not Cover a Full Year

**The constraint in the train/test split:**

1. The test period (April–October 2012) does **not include November or December** -> the model never evaluated against Thanksgiving or Christmas week sales, even though `step 3 already showed` that these are the highest-sales weeks of the year.

2. The dataset only spans ~2.7 years (Feb 2010 -> Oct 2012) (only **two** full holiday seasons in 2010 and 2011). Both `seasonal_decompose` and Holt-Winters (`seasonal_periods=52`) require **at least 2 full yearly cycles in the training data** to run at all. 
*So shifting the split will cause it to fail outright. -> the 2011 holiday season fell into the test set and training data would drop to (below the minimum the model needs, roughly 85-90 weeks)*

**A longer historical dataset (4–5 years) is needed**.

**What this means for interpreting results:**
- The MAE / RMSE / WMAE scores reported in Step 8 reflect how well the model forecasts **regular, non-holiday weeks** — they do **not** demonstrate how well it handles holiday-week spikes, since no holiday weeks exist in the test period at all.
- The WMAE metric (which weights holiday weeks 5x) is computed correctly, but in this case its value converges to plain MAE since there are effectively zero (or very few) holiday-flagged weeks in the test set to weight.
- Any claim about the model's holiday-forecasting accuracy would need to be validated separately — for example, with a longer dataset, or by manually holding out a holiday period from the *middle* of the data (an approach with its own trade-offs, since it breaks strict chronological ordering).

This limitation doesn't invalidate the analysis — it just means the scope of what's being validated should be stated accurately rather than implied to be broader than it is.

# 5. Trend & Seasonality Decomposition

**Process:** decompose the training series into *trend*, *seasonal*, and *residual* components using `statsmodels` to confirms whether the data has a repeating yearly pattern (for the train set only)

In [13]:
from statsmodels.tsa.seasonal import seasonal_decompose

decomposition = seasonal_decompose(train_ts, model='additive', period=52)

trend_component = decomposition.trend.dropna()
seasonal_component = decomposition.seasonal
resid_component = decomposition.resid.dropna()

print('Decomposition complete.')
print(f'Trend range     : {trend_component.min():,.0f}  to  {trend_component.max():,.0f}')
# trend range = max & min value from all the moving average of avg_weekly_sales
print(f'Seasonal swing  : {seasonal_component.min():,.0f}  to  {seasonal_component.max():,.0f}')
# seasonal swing = max & min value from all the value difference between the moving average and that particular week o   every year
print(f'Residual std dev: {resid_component.std():,.0f}')

Decomposition complete.
Trend range     : 46,739,591  to  47,531,238
Seasonal swing  : -6,994,004  to  34,288,373
Residual std dev: 293,013


### Observation result

- **Trend is nearly flat (~46.7M–47.5M)** -> steady baseline demand rather than organic growth or decline
- **Seasonal swing is large (-7M to +34M relative to trend)** -> holiday weeks don't just above the average, they represent a real large jump compared to the normal trendline
- **Residual std dev (~293K ) is small** (only ~0.6-0.7% of the ~40-50M weekly sales scale) -> trend + seasonality explain nearly all the variation, little unexplained noise

# 6. Baseline Model (Seasonal Naive)

**Process:** 
- Set a baseline -> a naive forecast that repeats **last year's value for the same week**

In [14]:
# Seasonal naive baseline: forecast = value from 52 weeks ago
naive_forecast = ts.shift(52).loc[test_ts.index]
# Safe from data leakage -> shifted 52 weeks back from the start and the end of the test set (Apr - Oct 2012) lands Apr - Oct 2011, which is the train set

# For weeks the shift(52) can't reach, fall back to the last known training value
naive_forecast = naive_forecast.ffill().fillna(train_ts.iloc[-1])

print('Baseline (seasonal naive) forecast -> first 5 weeks:')
print(naive_forecast.head())

Baseline (seasonal naive) forecast -> first 5 weeks:
Date
2012-04-13    44973328.14
2012-04-20    48676692.06
2012-04-27    43530032.78
2012-05-04    46861958.29
2012-05-11    45446144.82
Freq: W-FRI, Name: Total_Weekly_Sales, dtype: float64


*The comparison with test set will be done in step 8*

# 7. Forecasting Model: Holt-Winters Exponential Smoothing

**Holt-Winters models** trend and seasonality -> a solid fit for a weekly retail series with clear annual seasonality.

In [15]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

hw_model = ExponentialSmoothing(
    train_ts,
    trend='add',
    seasonal='add',
    seasonal_periods=52
).fit()
# There are 3 parameters ()

hw_forecast = hw_model.forecast(len(test_ts))
hw_forecast.index = test_ts.index

print('Holt-Winters forecast -> first 5 weeks:')
print(hw_forecast.head())

Holt-Winters forecast -> first 5 weeks:
Date
2012-04-13    4.710646e+07
2012-04-20    5.083798e+07
2012-04-27    4.570959e+07
2012-05-04    4.905862e+07
2012-05-11    4.765898e+07
Freq: W-FRI, dtype: float64


# 8. Model Evaluation

The baseline (seasonal naive) and Holt-Winters model comparison against actual test values based on:

- **MAE** (Mean Absolute Error)
- **RMSE** (Root Mean Squared Error)
- **WMAE** (Weighted MAE) -> Walmart's own competition metric (holiday weeks' weight 5x more than normal weeks)

In [16]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1) # guessing wrong on holiday week is more fatal
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

# Holiday flag aligned to test weeks (True if any store flagged that week as a holiday week)
holiday_flags = df.groupby('Date')['IsHoliday'].max().reindex(test_ts.index).fillna(False)

results = pd.DataFrame({
    'Model': ['Seasonal Naive (Baseline)', 'Holt-Winters'],
    'MAE': [
        mean_absolute_error(test_ts, naive_forecast),
        mean_absolute_error(test_ts, hw_forecast)
    ],
    'RMSE': [
        np.sqrt(mean_squared_error(test_ts, naive_forecast)),
        np.sqrt(mean_squared_error(test_ts, hw_forecast))
    ],
    'WMAE': [
        wmae(test_ts.values, naive_forecast.values, holiday_flags.values),
        wmae(test_ts.values, hw_forecast.values, holiday_flags.values)
    ]
}).round(0)

results

,Model,MAE,RMSE,WMAE
0,Seasonal Naive (Baseline),1247205.0,1562916.0,1285948.0
1,Holt-Winters,1825402.0,2192359.0,1676779.0


### Observation result

- `Holt-Winters` show higher MAE/RMSE/WMAE than `seasonal-naive baseline`.
- WMAE staying close to plain MAE -> model isn't failing specifically on holiday weeks
- If Holt-Winters ties or loses to the naive baseline, that's a sign the yearly pattern is extremely stable, but it will still worth re-checking on a longer historical window

# 9. Export Visual Dashboard to Excel

This project's visualizations are built as an **Excel workbook** using `openpyxl` 
1. an actual-vs-forecast chart
2. a seasonal decomposition chart
3. a store-type summary chart

In [17]:
import os
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.chart import LineChart, BarChart, Reference
from openpyxl.chart.marker import Marker
from openpyxl.utils import get_column_letter

os.makedirs('output', exist_ok=True)

# ---------- Color palette ----------
NAVY, ACCENT, GRAY, DARK, LIGHT_BG, GREEN, RED = \
    '1F3864', '2E75B6', '808080', '1F1F1F', 'F2F6FC', '2E8B57', 'C0504D'

HEADER_FILL = PatternFill('solid', start_color=NAVY, end_color=NAVY)
HEADER_FONT = Font(bold=True, color='FFFFFF', name='Calibri', size=11)
TITLE_FONT = Font(bold=True, size=18, name='Calibri', color=NAVY)
SUBTITLE_FONT = Font(size=11, name='Calibri', color='595959', italic=True)
KPI_LABEL_FONT = Font(size=10, name='Calibri', color='595959')
SECTION_FONT = Font(bold=True, size=13, name='Calibri', color=NAVY)
THIN = Side(style='thin', color='D9D9D9')
BORDER = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)

wb = Workbook()

# =========================================================
# SHEET: Dashboard (everything combined on ONE sheet)
# =========================================================
ws = wb.active
ws.title = 'Dashboard'
ws.sheet_view.showGridLines = False
for i, w in enumerate([2, 16, 16, 16, 16, 16, 2, 18, 18, 18, 18], start=1):
    ws.column_dimensions[get_column_letter(i)].width = w

ws['B2'] = 'Walmart Weekly Sales Forecast'
ws['B2'].font = TITLE_FONT
ws.merge_cells('B2:K2')
ws['B3'] = f'Holt-Winters vs Seasonal Naive Baseline  |  Test period: {test_ts.index.min().date()} to {test_ts.index.max().date()} ({len(test_ts)} weeks)'
ws['B3'].font = SUBTITLE_FONT
ws.merge_cells('B3:K3')

# ---- KPI cards ----
mae_naive, mae_hw = results.loc[0, 'MAE'], results.loc[1, 'MAE']
best_model = 'Seasonal Naive' if mae_naive < mae_hw else 'Holt-Winters'
best_mae = min(mae_naive, mae_hw)
improvement = abs(mae_naive - mae_hw) / max(mae_naive, mae_hw) * 100

kpis = [
    ('Best Model (by MAE)', best_model),
    ('Best Model MAE', f'${best_mae:,.0f}'),
    ('Improvement vs Other Model', f'{improvement:.0f}%'),
    ('Test Weeks Evaluated', f'{len(test_ts)}'),
]
for i, (label, value) in enumerate(kpis):
    c = 2 + i
    ws.cell(row=5, column=c, value=label).font = KPI_LABEL_FONT
    ws.cell(row=5, column=c).alignment = Alignment(wrap_text=True)
    ws.cell(row=6, column=c, value=value).font = Font(bold=True, size=16, color=NAVY)
    for r in (5, 6, 7):
        ws.cell(row=r, column=c).fill = PatternFill('solid', start_color=LIGHT_BG, end_color=LIGHT_BG)
        ws.cell(row=r, column=c).border = BORDER

# ---- Section 1: Actual vs Forecast (data + line chart) ----
ws['B9'] = 'Actual vs Forecasted Weekly Sales'
ws['B9'].font = SECTION_FONT

start_row = 10
ws.cell(row=start_row, column=2, value='Date')
ws.cell(row=start_row, column=3, value='Actual')
ws.cell(row=start_row, column=4, value='Seasonal Naive')
ws.cell(row=start_row, column=5, value='Holt-Winters')
for c in range(2, 6):
    ws.cell(row=start_row, column=c).font = HEADER_FONT
    ws.cell(row=start_row, column=c).fill = HEADER_FILL

for i, date in enumerate(test_ts.index):
    r = start_row + 1 + i
    ws.cell(row=r, column=2, value=date.strftime('%Y-%m-%d'))
    ws.cell(row=r, column=3, value=float(test_ts[date]))
    ws.cell(row=r, column=4, value=float(naive_forecast[date]))
    ws.cell(row=r, column=5, value=float(hw_forecast[date]))
n_rows = start_row + len(test_ts)

chart1 = LineChart()
chart1.y_axis.title = 'Weekly Sales ($)'
chart1.height, chart1.width = 8.5, 24
data = Reference(ws, min_col=3, max_col=5, min_row=start_row, max_row=n_rows)
cats = Reference(ws, min_col=2, min_row=start_row + 1, max_row=n_rows)
chart1.add_data(data, titles_from_data=True)
chart1.set_categories(cats)
for s, color, width, dash in zip(chart1.series, [DARK, GRAY, ACCENT], [22000, 12000, 22000], [None, 'dash', None]):
    s.graphicalProperties.line.solidFill = color
    s.graphicalProperties.line.width = width
    if dash:
        s.graphicalProperties.line.dashStyle = dash
    s.marker = Marker(symbol='none')
ws.add_chart(chart1, 'G9')

# ---- Section 2: Model Evaluation (data + bar chart) ----
GAP = 22  # minimum rows to leave room for a chart, even if the data table is short
eval_row = n_rows + 3
ws.cell(row=eval_row, column=2, value='Model Evaluation').font = SECTION_FONT
hdr_row = eval_row + 1
for c, name in zip(range(2, 6), ['Model', 'MAE', 'RMSE', 'WMAE']):
    ws.cell(row=hdr_row, column=c, value=name).font = HEADER_FONT
    ws.cell(row=hdr_row, column=c).fill = HEADER_FILL
for i, row in results.iterrows():
    r = hdr_row + 1 + i
    ws.cell(row=r, column=2, value=row['Model'])
    ws.cell(row=r, column=3, value=float(row['MAE']))
    ws.cell(row=r, column=4, value=float(row['RMSE']))
    ws.cell(row=r, column=5, value=float(row['WMAE']))
    for c in range(3, 6):
        ws.cell(row=r, column=c).number_format = '#,##0'

bar_eval = BarChart()
bar_eval.type = 'col'
bar_eval.title = 'Model Evaluation: MAE / RMSE / WMAE'
bar_eval.y_axis.title = 'Error ($)'
bar_eval.height, bar_eval.width = 8, 12
data = Reference(ws, min_col=3, max_col=5, min_row=hdr_row, max_row=hdr_row + len(results))
cats = Reference(ws, min_col=2, min_row=hdr_row + 1, max_row=hdr_row + len(results))
bar_eval.add_data(data, titles_from_data=True)
bar_eval.set_categories(cats)
for s, color in zip(bar_eval.series, [GRAY, RED, ACCENT]):
    s.graphicalProperties.solidFill = color
ws.add_chart(bar_eval, 'G' + str(eval_row))

# ---- Section 3: Store Type Summary (data + bar chart) ----
type_reset = type_summary.reset_index()
type_row = hdr_row + max(len(results) + 3, GAP)
ws.cell(row=type_row, column=2, value='Average Weekly Sales by Store Type').font = SECTION_FONT
thdr = type_row + 1
ws.cell(row=thdr, column=2, value='Type').font = HEADER_FONT
ws.cell(row=thdr, column=2).fill = HEADER_FILL
ws.cell(row=thdr, column=3, value='Avg_Weekly_Sales').font = HEADER_FONT
ws.cell(row=thdr, column=3).fill = HEADER_FILL
for i, row in type_reset.iterrows():
    r = thdr + 1 + i
    ws.cell(row=r, column=2, value=row['Type'])
    ws.cell(row=r, column=3, value=float(row['Avg_Weekly_Sales']))

bar_type = BarChart()
bar_type.type = 'col'
bar_type.title = 'Average Weekly Sales by Store Type'
bar_type.y_axis.title = 'Avg Weekly Sales ($)'
bar_type.height, bar_type.width = 8, 12
data = Reference(ws, min_col=3, min_row=thdr, max_row=thdr + len(type_reset))
cats = Reference(ws, min_col=2, min_row=thdr + 1, max_row=thdr + len(type_reset))
bar_type.add_data(data, titles_from_data=True)
bar_type.set_categories(cats)
bar_type.series[0].graphicalProperties.solidFill = ACCENT
bar_type.legend = None
ws.add_chart(bar_type, 'G' + str(type_row))

# ---- Section 4: Trend & Seasonal Components (data + line chart) ----
decomp_df_export = pd.DataFrame({
    'Trend': decomposition.trend,
    'Seasonal': decomposition.seasonal
}).dropna()

decomp_row = thdr + max(len(type_reset) + 3, GAP)
ws.cell(row=decomp_row, column=2, value='Trend & Seasonal Components').font = SECTION_FONT
dhdr = decomp_row + 1
for c, name in zip(range(2, 5), ['Date', 'Trend', 'Seasonal']):
    ws.cell(row=dhdr, column=c, value=name).font = HEADER_FONT
    ws.cell(row=dhdr, column=c).fill = HEADER_FILL
for i, (date, row) in enumerate(decomp_df_export.iterrows()):
    r = dhdr + 1 + i
    ws.cell(row=r, column=2, value=date.strftime('%Y-%m-%d'))
    ws.cell(row=r, column=3, value=float(row['Trend']))
    ws.cell(row=r, column=4, value=float(row['Seasonal']))
n_d = dhdr + len(decomp_df_export)

chart_d = LineChart()
chart_d.title = 'Trend & Seasonal Components'
chart_d.height, chart_d.width = 8.5, 24
data = Reference(ws, min_col=3, max_col=4, min_row=dhdr, max_row=n_d)
cats = Reference(ws, min_col=2, min_row=dhdr + 1, max_row=n_d)
chart_d.add_data(data, titles_from_data=True)
chart_d.set_categories(cats)
for s, color in zip(chart_d.series, [NAVY, ACCENT]):
    s.graphicalProperties.line.solidFill = color
    s.graphicalProperties.line.width = 18000
    s.marker = Marker(symbol='none')
ws.add_chart(chart_d, 'G' + str(decomp_row))

wb.save('output/Walmart_Sales_Forecast_Dashboard.xlsx')